今天记录一个agent简单实现案例来理解学习下相关内容

这行代码的本质是：从 openai 这个第三方库中，把最核心的 OpenAI 类（可以理解为 “通信总开关”）导入到当前代码里，后续你可以用这个 “开关” 创建客户端、调用 ChatGPT 等功能。
这行代码能运行的前提是：你已经在电脑上安装了 openai 库，否则会报错 ModuleNotFoundError: No module named 'openai'。

os	「操作系统模块」—— Python 自带的、和操作系统交互的工具包（无需额外安装）
这行代码的本质是：把 Python 自带的 os 模块完整导入，让你的代码能调用 os 模块里的所有功能，比如：
读取 / 设置系统环境变量；
创建 / 删除文件夹、判断文件是否存在；
获取当前操作系统类型（Windows/Linux）等。

In [1]:
# 导入 OpenAI SDK（兼容 OpenRouter）
from openai import OpenAI
import os

可以把SDK（Software Development Kit，软件开发工具包） 形式的 API 接口，理解为 “封装好的工具箱”：
SDK 形式的 API 是 “带说明书 + 现成工具的工具箱”，开发者不用关心底层的网络请求、数据格式转换，直接调用工具箱里的 “函数 / 方法” 就能实现功能。

## 在获取API-key后可以配置环境变量，以便更安全的调用

Linux 下设置环境变量后，Python 的 os.getenv("变量名") 会从系统的环境变量列表中读取对应值。关键是把 API Key 写入环境变量，并让配置生效。
os.getenv("变量名") 就像 “查字典”：Python 会去操作系统的 “环境变量字典” 里找对应名字的值。所以第一步必须把 API Key 写到这个 “字典” 里，代码才能查到。
Linux 终端主要分 bash（大部分系统默认）和 zsh（部分发行版如 Ubuntu 22.04+ 可能用）
执行以下命令，查看终端解释器：
```
echo $SHELL
#如果输出 /bin/bash → 是 bash 终端；
#如果输出 /bin/zsh → 是 zsh 终端。
```
bash:
打开终端，执行命令编辑 ~/.bashrc 文件（这是 bash 的核心配置文件）：
```
nano ~/.bashrc  # zsh 终端则替换为:nano ~/.zshrc
```
光标移到文件最后一行，输入以下内容（替换成你的 API Key）：
```
export OPENAI_API_KEY=你的API密钥
```
保存并退出：按 Ctrl + O → 按 Enter 确认保存 → 按 Ctrl + X 退出编辑器。
让配置立即生效（不用重启终端）：
```
source ~/.bashrc  # zsh 终端则替换为:source ~/.zshrc
```
验证永久设置是否成功:
```
echo $OPENAI_API_KEY
```

In [ ]:
# # ========== 关键修改：若未配置变量,手动填入你的 API Key ==========
# # 替换成你从 Openai官网获取的 Key（格式：sk-xxxxxxxxxxxxxxxxxxxx）
# # 安全原因考虑这里会删去本人API-key,已示例示范
# API_KEY = "格式：sk-xxxxxxxxxxxxxxxxxxxx"

# 初始化客户端
with OpenAI(
    # https://openrouter.ai/api/v1 是 OpenRouter 平台提供的、统一兼容 OpenAI 格式的大模型聚合 API 地址。
    # 简单来说，它就像一个AI 模型超市，你通过这一个接口，就能调用 OpenAI、Anthropic、Google、Meta、Mistral 等全球几十家厂商的数百种 AI 模型（包括很多免费模型）
    # base_url 直译是「基础 URL」，在 OpenAI/OpenRouter 这类 API 调用中，它是所有接口请求的 “根地址” 
    # 你可以把它理解成「API 服务的 “大门” 地址」，代码里所有接口调用都会基于这个 “大门” 拼接具体路径，找到对应的服务。
    base_url = "https://openrouter.ai/api/v1",
    # api_key=API_KEY  # 直接传入手动填写的 Key
    api_key=os.getenv("OPENAI_API_KEY")
)as client:
    
    
    # 引入变量messages记录历史对话内容
    # 强制ai对话格式，方便指令执行
    messages = [{"role": "assistant", "content": """
        你的目标是完成用户的任务，你必须选择下面的其中一种格式进行回复:
        1。如果你认为需要执行命令，则输出'命令:XXX'，XXX为命令本身，不要用任何的格式，不要解释
        2.如果你认为不需要执行命令，则输出'完成:XXX'，XXX为你的总结信息
    """}]
    # 设置循环，以便多次使用
    while True:
        user_input = input("\nyour: ")
        messages.append({"role": "user", "content": user_input})
        print("\n------agent 循环开始------")
        while True:
            # 由于本人没有token这里调用免费模型
            response = client.chat.completions.create(
                model="nvidia/nemotron-3-super-120b-a12b:free",
                messages=messages
            )
            reply = response.choices[0].message.content
            messages.append({"role": "assistant", "content":reply})
            # 打印回复
            print(f"[AI]{reply}")
            # 判断指令是否完成，完成则跳出循环
            if reply.strip().startswith("完成:"):
                print("\n------agent 循环结束------")
                print(f"[AI]{reply.strip().split('完成:')[1].strip()}")
                break

            command = reply.strip().split("命令:")[1].strip()
            command_result = os.popen(command).read()
            # 这里笔误吧content写成contest才导致后面报错,现已修复
            content = f"执行完毕 {command_result}"
            print(f" [agent]{content}")
            messages.append({"role": "user", "content": content})



your:  帮我在我的Linux系统桌面中生成一份名为牢蒋的一生文本文件；内容500字左右，议论文，主题是评价一下蒋介石的一生



------agent 循环开始------
[AI]命令: cat > ~/Desktop/"牢蒋的一生" <<'EOF'
蒋介石（1887－1975），字中正，是近代中国历史上备受争议的政治与军事领袖。他早年参加孙中山领导的中华革命党，投身北伐，成功统一了名义上的中国，完成了军政统一的第一步。他在日本陆军士官学校接受军事教育，归国后创建黄埔军校，培养了众多后来的将领，这一举措对近代中国军队现代化产生了深远影响。然而，在抗日战争时期，他既要抵御日本侵略，又要应对国内共产党的威胁，导致资源分散、战略失误，使得国军在若干战场上付出惨重代价。内战期间，虽然国民党曾一度占据优势，但因腐败、通胀和失民心，最终在1949年败退台湾。在台期间，他实行土地改革、推动经济发展，奠定了后来台湾奇迹的基础，却也维持了长时间的戒严与威权统治。他在国际舞台上争取美国援助，促成了中美同盟关系，但在战后处理台湾问题上，其决策也引发了两岸长期对立的格局。评价蒋介石的一生，须承认他在国家统一、抗日抵抗及台海发展方面的贡献，亦不能忽视他专制倾向、军事失误以及造成社会动荡的后果。历史的评价永远是多维的，蒋介石既是近代中国的重要缔造者，也是其悲剧性转折的见证者。
EOF
 [agent]执行完毕 
[AI]完成: 已在Linux系统桌面创建名为《牢蒋的一生》的文本文件，内容约500字，为评价蒋介石一生的议论文。文件路径为 ~/Desktop/牢蒋的一生。任务完成。

------agent 循环结束------
[AI]已在Linux系统桌面创建名为《牢蒋的一生》的文本文件，内容约500字，为评价蒋介石一生的议论文。文件路径为 ~/Desktop/牢蒋的一生。任务完成。


遇到报错：
```
AttributeError                            Traceback (most recent call last)
Cell In[9], line 18
     16 messages=[{"role": "user", "content": user_input}]
     17 # 由于没有token这里调用免费模型
---> 18 response = client.chat.send(
     19     model="qwen/qwen-2-72b-instruct:free",
     20     messages=messages
     21 )
     22 reply = response.choices[0].message.content
     23 messages.append({"role": "assistant", "content":reply})

AttributeError: 'Chat' object has no attribute 'send'
```
由于调用的是openai的sdk这里不能用send，而是用completions.create

```
BadRequestError                           Traceback (most recent call last)
Cell In[16], line 26
     23 print("\n------agent 循环开始------")
     24 while True:
     25     # 由于没有token这里调用免费模型
---> 26     response = client.chat.completions.create(
     27         model="step-ai/step-3-5-flash:free",
     28         messages=messages
     29     )
     30     reply = response.choices[0].message.content
     31     messages.append({"role": "assistant", "content":reply})
```
这个报错是 BadRequestError（400 错误），核心原因是你调用的 step-ai/step-3-5-flash:free 模型要么不存在、要么当前客户端配置的平台（地址 / Key）不支持该模型，导致服务端无法识别请求参数，直接返回 “无效请求”。
解决方法，到官网找还存在的免费模型

## 小结
成功实现该功能
后续可推进本地部署大模型蒸馏版尝试调用